
# Feature Engineering

## Credit Risk Intelligence Lab

This notebook transforms the exploratory credit dataset into a clean and reusable modeling dataset.

The purpose of this stage is to move from descriptive analysis to a structured machine learning pipeline. In credit risk, feature engineering is not only a technical step; it is the bridge between raw borrower information and interpretable risk signals.

This notebook prepares the data for two later modeling directions: supervised learning, where the objective is to estimate probability of default, and unsupervised learning, where the objective is to discover hidden borrower segments and anomalous credit profiles.



## Notebook objectives

This notebook focuses on six tasks:

1. Encode categorical variables.
2. Scale numerical variables when needed.
3. Create financial risk features such as credit amount per month, age buckets, duration buckets, and exposure proxies.
4. Prepare a clean modeling dataset.
5. Define a reproducible train-test split.
6. Build a preprocessing pipeline that can be reused by supervised and unsupervised models.


In [1]:

from pathlib import Path
import warnings
import pickle

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 150)
pd.set_option("display.width", 150)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")



## Project paths

The notebook is designed to run either from the project root or from the `notebooks/` directory.


In [2]:

# Detect project root
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"

for path in [DATA_RAW, DATA_PROCESSED, MODELS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data path: {DATA_RAW}")
print(f"Processed data path: {DATA_PROCESSED}")
print(f"Models path: {MODELS_DIR}")


Project root: /Users/andremgx/Documents/GitHub/credit-risk-intelligence-lab
Raw data path: /Users/andremgx/Documents/GitHub/credit-risk-intelligence-lab/data/raw
Processed data path: /Users/andremgx/Documents/GitHub/credit-risk-intelligence-lab/data/processed
Models path: /Users/andremgx/Documents/GitHub/credit-risk-intelligence-lab/models



## Load analytical dataset

This notebook first looks for the dataset created in the exploratory analysis notebook:

```text
data/processed/german_credit_eda.csv
```

If that file does not exist, the notebook falls back to the raw German Credit Data file. If the raw file is also missing, the notebook downloads the dataset and reconstructs the target variable.


In [3]:

column_names = [
    "checking_account_status",
    "duration_months",
    "credit_history",
    "purpose",
    "credit_amount",
    "savings_account",
    "employment_since",
    "installment_rate_pct_income",
    "personal_status_sex",
    "other_debtors_guarantors",
    "present_residence_since",
    "property",
    "age_years",
    "other_installment_plans",
    "housing",
    "existing_credits",
    "job",
    "people_liable",
    "telephone",
    "foreign_worker",
    "target_original",
]

eda_file = DATA_PROCESSED / "german_credit_eda.csv"
raw_file = DATA_RAW / "german_credit_raw.csv"

if eda_file.exists():
    df = pd.read_csv(eda_file)
    print("Loaded EDA dataset from data/processed.")
elif raw_file.exists():
    df = pd.read_csv(raw_file)
    print("Loaded raw dataset from data/raw.")
else:
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"
    df = pd.read_csv(url, sep=" ", header=None, names=column_names)
    df.to_csv(raw_file, index=False)
    print("Downloaded raw German Credit Data.")

if "default" not in df.columns:
    df["default"] = np.where(df["target_original"] == 2, 1, 0)

df.head()


Loaded EDA dataset from data/processed.


,checking_account_status,duration_months,credit_history,purpose,credit_amount,savings_account,employment_since,installment_rate_pct_income,personal_status_sex,other_debtors_guarantors,present_residence_since,property,age_years,other_installment_plans,housing,existing_credits,job,people_liable,telephone,foreign_worker,target_original,default,credit_amount_bucket,duration_bucket
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,4,A121,67,A143,A152,2,A173,1,A192,A201,1,0,low_amount,short_duration
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,2,A121,22,A143,A152,1,A173,1,A191,A201,2,1,high_amount,long_duration
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,3,A121,49,A143,A152,1,A172,2,A191,A201,1,0,medium_low_amount,short_duration
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,4,A122,45,A143,A153,1,A173,2,A191,A201,1,0,high_amount,long_duration
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,4,A124,53,A143,A153,2,A173,2,A191,A201,2,1,high_amount,medium_long_duration


In [4]:

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

df.info()


Rows: 1,000
Columns: 24
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 24 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   checking_account_status      1000 non-null   object
 1   duration_months              1000 non-null   int64 
 2   credit_history               1000 non-null   object
 3   purpose                      1000 non-null   object
 4   credit_amount                1000 non-null   int64 
 5   savings_account              1000 non-null   object
 6   employment_since             1000 non-null   object
 7   installment_rate_pct_income  1000 non-null   int64 
 8   personal_status_sex          1000 non-null   object
 9   other_debtors_guarantors     1000 non-null   object
 10  present_residence_since      1000 non-null   int64 
 11  property                     1000 non-null   object
 12  age_years                    1000 non-null   int64 
 13  other_inst


## Target variable

The target variable is:

```text
default = 1  → bad credit risk
default = 0  → good credit risk
```

This convention is useful because the model output can later be interpreted as a probability of default.


In [5]:

target_col = "default"

target_summary = (
    df[target_col]
    .value_counts()
    .sort_index()
    .to_frame("count")
    .assign(rate=lambda x: x["count"] / x["count"].sum())
)

target_summary.index = ["good_credit", "bad_credit"]  # type: ignore
target_summary


,count,rate
good_credit,700,0.7000
bad_credit,300,0.3000



## Remove exploratory-only columns

The first EDA notebook may have created exploratory bucket variables. In this notebook, we rebuild engineered variables from scratch to avoid accidental leakage or duplicated logic.


In [6]:

exploratory_columns = ["credit_amount_bucket", "duration_bucket"]

df_model = df.drop(columns=[col for col in exploratory_columns if col in df.columns]).copy()
df_model.head()


,checking_account_status,duration_months,credit_history,purpose,credit_amount,savings_account,employment_since,installment_rate_pct_income,personal_status_sex,other_debtors_guarantors,present_residence_since,property,age_years,other_installment_plans,housing,existing_credits,job,people_liable,telephone,foreign_worker,target_original,default
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,4,A121,67,A143,A152,2,A173,1,A192,A201,1,0
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,2,A121,22,A143,A152,1,A173,1,A191,A201,2,1
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,3,A121,49,A143,A152,1,A172,2,A191,A201,1,0
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,4,A122,45,A143,A153,1,A173,2,A191,A201,1,0
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,4,A124,53,A143,A153,2,A173,2,A191,A201,2,1



## Financial feature engineering

This section creates interpretable credit risk features.

The goal is not to create dozens of artificial variables, but to build a compact set of features that make financial sense: credit intensity, maturity, exposure level, borrower lifecycle, and a simple repayment burden proxy.


In [7]:

df_fe = df_model.copy()

df_fe["credit_amount_per_month"] = df_fe["credit_amount"] / df_fe["duration_months"].replace(0, np.nan)
df_fe["credit_amount_log"] = np.log1p(df_fe["credit_amount"])
df_fe["duration_years"] = df_fe["duration_months"] / 12
df_fe["long_duration_flag"] = np.where(df_fe["duration_months"] >= 36, 1, 0)

high_credit_threshold = df_fe["credit_amount"].quantile(0.75)
df_fe["high_credit_amount_flag"] = np.where(df_fe["credit_amount"] >= high_credit_threshold, 1, 0)

df_fe["installment_burden_proxy"] = (
    df_fe["installment_rate_pct_income"] * df_fe["credit_amount"]
)

df_fe["age_bucket"] = pd.cut(
    df_fe["age_years"],
    bins=[0, 25, 35, 50, np.inf],
    labels=["young", "early_career", "mid_career", "senior"],
    right=True,
)

df_fe["duration_bucket"] = pd.cut(
    df_fe["duration_months"],
    bins=[0, 12, 24, 36, np.inf],
    labels=["short_term", "medium_term", "long_term", "very_long_term"],
    right=True,
)

df_fe["credit_amount_bucket"] = pd.qcut(
    df_fe["credit_amount"],
    q=4,
    labels=["low_amount", "medium_low_amount", "medium_high_amount", "high_amount"],
    duplicates="drop",
)

df_fe.head()


,checking_account_status,duration_months,credit_history,purpose,credit_amount,savings_account,employment_since,installment_rate_pct_income,personal_status_sex,other_debtors_guarantors,present_residence_since,property,age_years,other_installment_plans,housing,existing_credits,job,people_liable,telephone,foreign_worker,target_original,default,credit_amount_per_month,credit_amount_log,duration_years,long_duration_flag,high_credit_amount_flag,installment_burden_proxy,age_bucket,duration_bucket,credit_amount_bucket
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,4,A121,67,A143,A152,2,A173,1,A192,A201,1,0,194.8333,7.0648,0.5000,0,0,4676,senior,short_term,low_amount
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,2,A121,22,A143,A152,1,A173,1,A191,A201,2,1,123.9792,8.6915,4.0000,1,1,11902,young,very_long_term,high_amount
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,3,A121,49,A143,A152,1,A172,2,A191,A201,1,0,174.6667,7.6483,1.0000,0,0,4192,mid_career,short_term,medium_low_amount
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,4,A122,45,A143,A153,1,A173,2,A191,A201,1,0,187.6667,8.9725,3.5000,1,1,15764,mid_career,very_long_term,high_amount
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,4,A124,53,A143,A153,2,A173,2,A191,A201,2,1,202.9167,8.4911,2.0000,0,1,14610,senior,medium_term,high_amount



## Validate engineered features

After creating features, we check whether the new variables were created correctly and whether they contain missing values.


In [8]:

engineered_features = [
    "credit_amount_per_month",
    "credit_amount_log",
    "duration_years",
    "long_duration_flag",
    "high_credit_amount_flag",
    "installment_burden_proxy",
    "age_bucket",
    "duration_bucket",
    "credit_amount_bucket",
]

df_fe[engineered_features].head()


,credit_amount_per_month,credit_amount_log,duration_years,long_duration_flag,high_credit_amount_flag,installment_burden_proxy,age_bucket,duration_bucket,credit_amount_bucket
0,194.8333,7.0648,0.5000,0,0,4676,senior,short_term,low_amount
1,123.9792,8.6915,4.0000,1,1,11902,young,very_long_term,high_amount
2,174.6667,7.6483,1.0000,0,0,4192,mid_career,short_term,medium_low_amount
3,187.6667,8.9725,3.5000,1,1,15764,mid_career,very_long_term,high_amount
4,202.9167,8.4911,2.0000,0,1,14610,senior,medium_term,high_amount


In [9]:

feature_quality_report = (
    df_fe[engineered_features]
    .isna()
    .sum()
    .to_frame("missing_count")
    .assign(missing_rate=lambda x: x["missing_count"] / len(df_fe))
    .sort_values("missing_count", ascending=False)
)

feature_quality_report


,missing_count,missing_rate
credit_amount_per_month,0,0.0000
credit_amount_log,0,0.0000
duration_years,0,0.0000
long_duration_flag,0,0.0000
high_credit_amount_flag,0,0.0000
installment_burden_proxy,0,0.0000
age_bucket,0,0.0000
duration_bucket,0,0.0000
credit_amount_bucket,0,0.0000



## Risk signal review of engineered categorical features

Before modeling, we verify whether the engineered categorical features show meaningful differences in default rates. This does not prove causality, but it helps determine whether the engineered variables are useful for risk segmentation.


In [10]:

def categorical_risk_summary(data: pd.DataFrame, feature: str, target: str = "default") -> pd.DataFrame:
    summary = (
        data
        .groupby(feature, observed=False)
        .agg(
            borrowers=(target, "size"),
            defaults=(target, "sum"),
            default_rate=(target, "mean"),
        )
        .assign(portfolio_share=lambda x: x["borrowers"] / len(data))
        .sort_values("default_rate", ascending=False)
    )
    return summary

for feature in ["age_bucket", "duration_bucket", "credit_amount_bucket"]:
    print(f"\nFeature: {feature}")
    display(categorical_risk_summary(df_fe, feature, target_col))



Feature: age_bucket


,borrowers,defaults,default_rate,portfolio_share
age_bucket,,,,
young,190,80,0.4211,0.1900
early_career,398,118,0.2965,0.3980
senior,113,31,0.2743,0.1130
mid_career,299,71,0.2375,0.2990



Feature: duration_bucket


,borrowers,defaults,default_rate,portfolio_share
duration_bucket,,,,
very_long_term,87,45,0.5172,0.0870
long_term,143,57,0.3986,0.1430
medium_term,411,122,0.2968,0.4110
short_term,359,76,0.2117,0.3590



Feature: credit_amount_bucket


,borrowers,defaults,default_rate,portfolio_share
credit_amount_bucket,,,,
high_amount,250,105,0.4200,0.2500
low_amount,250,77,0.3080,0.2500
medium_low_amount,250,62,0.2480,0.2500
medium_high_amount,250,56,0.2240,0.2500



## Define modeling columns

We now separate the target from the input variables. The original target column `target_original` is removed to avoid leakage because it is only another representation of the same target.


In [11]:

columns_to_drop = ["target_original", target_col]

X = df_fe.drop(columns=[col for col in columns_to_drop if col in df_fe.columns])
y = df_fe[target_col].copy()

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")


X shape: (1000, 29)
y shape: (1000,)



## Identify numerical and categorical features

The preprocessing pipeline will apply different transformations depending on the feature type. Numerical variables will be scaled and categorical variables will be one-hot encoded.

Scaling is especially useful for distance-based methods such as PCA, K-Means, HDBSCAN-style workflows and anomaly detection.


In [12]:

numerical_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

print(f"Numerical features ({len(numerical_features)}):")
print(numerical_features)

print(f"\nCategorical features ({len(categorical_features)}):")
print(categorical_features)


Numerical features (13):
['duration_months', 'credit_amount', 'installment_rate_pct_income', 'present_residence_since', 'age_years', 'existing_credits', 'people_liable', 'credit_amount_per_month', 'credit_amount_log', 'duration_years', 'long_duration_flag', 'high_credit_amount_flag', 'installment_burden_proxy']

Categorical features (16):
['checking_account_status', 'credit_history', 'purpose', 'savings_account', 'employment_since', 'personal_status_sex', 'other_debtors_guarantors', 'property', 'other_installment_plans', 'housing', 'job', 'telephone', 'foreign_worker', 'age_bucket', 'duration_bucket', 'credit_amount_bucket']



## Train-test split

We create a reproducible train-test split using stratification. Stratification preserves the target distribution across training and testing sets, which is important in credit risk because the default class is often less frequent than the non-default class.


In [13]:

RANDOM_STATE = 42
TEST_SIZE = 0.20

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}")

print("\nTrain default rate:", round(y_train.mean(), 4))
print("Test default rate:", round(y_test.mean(), 4))


X_train: (800, 29)
X_test: (200, 29)
y_train: (800,)
y_test: (200,)

Train default rate: 0.3
Test default rate: 0.3



## Build preprocessing pipeline

The preprocessing object created here will be reused in later notebooks.

This is a key design decision: instead of manually encoding and scaling variables in each notebook, we define a formal preprocessing pipeline. This makes the project more reproducible and closer to professional machine learning practice.


In [14]:

try:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("onehot", one_hot_encoder),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numerical_features),
        ("cat", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
)

preprocessor


,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,copy,True
,with_mean,True
,with_std,True



## Fit preprocessing pipeline

The preprocessor must be fitted only on the training set. The test set must be transformed using the fitted training preprocessor. This avoids data leakage from the test set into the training process.


In [15]:

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"Processed X_train shape: {X_train_processed.shape}")
print(f"Processed X_test shape: {X_test_processed.shape}")


Processed X_train shape: (800, 79)
Processed X_test shape: (200, 79)



## Recover processed feature names

Recovering feature names is essential for interpretability. Later, these names will be used to inspect model coefficients, feature importance, SHAP values, and cluster profiles.


In [16]:

def get_processed_feature_names(preprocessor: ColumnTransformer) -> list:
    feature_names = []
    feature_names.extend(numerical_features)

    cat_transformer = preprocessor.named_transformers_["cat"]
    encoder = cat_transformer.named_steps["onehot"]
    cat_feature_names = encoder.get_feature_names_out(categorical_features).tolist()
    feature_names.extend(cat_feature_names)

    return feature_names

processed_feature_names = get_processed_feature_names(preprocessor)

print(f"Number of processed features: {len(processed_feature_names)}")
processed_feature_names[:20]


Number of processed features: 79


['duration_months',
 'credit_amount',
 'installment_rate_pct_income',
 'present_residence_since',
 'age_years',
 'existing_credits',
 'people_liable',
 'credit_amount_per_month',
 'credit_amount_log',
 'duration_years',
 'long_duration_flag',
 'high_credit_amount_flag',
 'installment_burden_proxy',
 'checking_account_status_A11',
 'checking_account_status_A12',
 'checking_account_status_A13',
 'checking_account_status_A14',
 'credit_history_A30',
 'credit_history_A31',
 'credit_history_A32']

In [17]:
X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=processed_feature_names,
    index=X_train.index,
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=processed_feature_names,
    index=X_test.index,
)

X_train_processed_df.head()

,duration_months,credit_amount,installment_rate_pct_income,present_residence_since,age_years,existing_credits,people_liable,credit_amount_per_month,credit_amount_log,duration_years,long_duration_flag,high_credit_amount_flag,installment_burden_proxy,checking_account_status_A11,checking_account_status_A12,checking_account_status_A13,checking_account_status_A14,credit_history_A30,credit_history_A31,credit_history_A32,credit_history_A33,credit_history_A34,purpose_A40,purpose_A41,purpose_A410,purpose_A42,purpose_A43,purpose_A44,purpose_A45,purpose_A46,purpose_A48,purpose_A49,savings_account_A61,savings_account_A62,savings_account_A63,savings_account_A64,savings_account_A65,employment_since_A71,employment_since_A72,employment_since_A73,employment_since_A74,employment_since_A75,personal_status_sex_A91,personal_status_sex_A92,personal_status_sex_A93,personal_status_sex_A94,other_debtors_guarantors_A101,other_debtors_guarantors_A102,other_debtors_guarantors_A103,property_A121,property_A122,property_A123,property_A124,other_installment_plans_A141,other_installment_plans_A142,other_installment_plans_A143,housing_A151,housing_A152,housing_A153,job_A171,job_A172,job_A173,job_A174,telephone_A191,telephone_A192,foreign_worker_A201,foreign_worker_A202,age_bucket_early_career,age_bucket_mid_career,age_bucket_senior,age_bucket_young,duration_bucket_long_term,duration_bucket_medium_term,duration_bucket_short_term,duration_bucket_very_long_term,credit_amount_bucket_high_amount,credit_amount_bucket_low_amount,credit_amount_bucket_medium_high_amount,credit_amount_bucket_medium_low_amount
828,1.2896,1.9258,0.0523,1.0534,1.0578,-0.7187,-0.4364,0.5928,1.6322,1.2896,2.2194,1.7614,2.0515,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,1.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,1.0000,0.0000,0.0000,1.0000,0.0000,0.0000,1.0000,0.0000,1.0000,0.0000,1.0000,0.0000,0.0000,1.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000
997,-0.7426,-0.8929,0.9422,1.0534,0.2424,-0.7187,-0.4364,-0.8210,-1.4040,-0.7426,-0.4506,-0.5677,-0.6830,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,1.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,1.0000,0.0000,1.0000,0.0000,0.0000,0.0000,1.0000,0.0000,1.0000,0.0000,1.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,1.0000,0.0000,0.0000
148,1.2896,0.8164,0.0523,-0.7473,-0.6635,1.0450,-0.4364,-0.1147,1.0614,1.2896,2.2194,1.7614,0.9356,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,1.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,1.0000,0.0000,0.0000,0.0000,1.0000,0.0000,1.0000,0.0000,1.0000,0.0000,1.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000
735,1.2896,0.2996,0.0523,-0.7473,-0.5729,-0.7187,-0.4364,-0.4443,0.6755,1.2896,2.2194,1.7614,0.4156,0.0000,1.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,1.0000,0.0000,0.0000,0.0000,1.0000,0.0000,1.0000,0.0000,0.0000,0.0000,1.0000,0.0000,1.0000,0.0000,1.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000
130,2.3057,1.9827,-1.7275,-0.7473,-1.0258,-0.7187,-0.4364,0.1226,1.6556,2.3057,2.2194,1.7614,-0.0215,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0


## Save modeling datasets

The raw train-test files preserve the original interpretable columns. The processed files are ready for machine learning algorithms.


In [18]:

X_train.to_csv(DATA_PROCESSED / "X_train.csv", index=True)
X_test.to_csv(DATA_PROCESSED / "X_test.csv", index=True)
y_train.to_csv(DATA_PROCESSED / "y_train.csv", index=True)
y_test.to_csv(DATA_PROCESSED / "y_test.csv", index=True)

X_train_processed_df.to_csv(DATA_PROCESSED / "X_train_processed.csv", index=True)
X_test_processed_df.to_csv(DATA_PROCESSED / "X_test_processed.csv", index=True)

df_fe.to_csv(DATA_PROCESSED / "german_credit_features.csv", index=False)

with open(MODELS_DIR / "preprocessor.pkl", "wb") as f:
    pickle.dump(preprocessor, f)

print("Saved train-test splits, processed matrices, engineered dataset, and preprocessing pipeline.")


Saved train-test splits, processed matrices, engineered dataset, and preprocessing pipeline.



## Create a feature dictionary

A feature dictionary helps document the meaning of each engineered variable. This is useful for GitHub readers, model governance, and future README documentation.


In [19]:

feature_dictionary = pd.DataFrame(
    [
        {
            "feature": "credit_amount_per_month",
            "type": "engineered_numerical",
            "description": "Credit amount divided by loan duration in months. Proxy for monthly credit intensity.",
        },
        {
            "feature": "credit_amount_log",
            "type": "engineered_numerical",
            "description": "Log transformation of credit amount to reduce skewness.",
        },
        {
            "feature": "duration_years",
            "type": "engineered_numerical",
            "description": "Loan duration expressed in years.",
        },
        {
            "feature": "long_duration_flag",
            "type": "engineered_binary",
            "description": "Indicator equal to 1 when loan duration is 36 months or longer.",
        },
        {
            "feature": "high_credit_amount_flag",
            "type": "engineered_binary",
            "description": "Indicator equal to 1 when credit amount is above or equal to the 75th percentile.",
        },
        {
            "feature": "installment_burden_proxy",
            "type": "engineered_numerical",
            "description": "Installment rate multiplied by credit amount. Proxy for repayment burden.",
        },
        {
            "feature": "age_bucket",
            "type": "engineered_categorical",
            "description": "Borrower lifecycle group based on age.",
        },
        {
            "feature": "duration_bucket",
            "type": "engineered_categorical",
            "description": "Loan maturity group based on duration in months.",
        },
        {
            "feature": "credit_amount_bucket",
            "type": "engineered_categorical",
            "description": "Exposure group based on credit amount quartiles.",
        },
    ]
)

feature_dictionary.to_csv(DATA_PROCESSED / "feature_dictionary.csv", index=False)

feature_dictionary


,feature,type,description
0,credit_amount_per_month,engineered_numerical,Credit amount divided by loan duration in mont...
1,credit_amount_log,engineered_numerical,Log transformation of credit amount to reduce ...
2,duration_years,engineered_numerical,Loan duration expressed in years.
3,long_duration_flag,engineered_binary,Indicator equal to 1 when loan duration is 36 ...
4,high_credit_amount_flag,engineered_binary,Indicator equal to 1 when credit amount is abo...
5,installment_burden_proxy,engineered_numerical,Installment rate multiplied by credit amount. ...
6,age_bucket,engineered_categorical,Borrower lifecycle group based on age.
7,duration_bucket,engineered_categorical,Loan maturity group based on duration in months.
8,credit_amount_bucket,engineered_categorical,Exposure group based on credit amount quartiles.



## Modeling readiness checklist

Before moving to supervised modeling, we verify the minimum conditions required for a clean baseline model.


In [20]:

readiness_check = {
    "missing_values_X_train_processed": int(X_train_processed_df.isna().sum().sum()),
    "missing_values_X_test_processed": int(X_test_processed_df.isna().sum().sum()),
    "n_features_train": X_train_processed_df.shape[1],
    "n_features_test": X_test_processed_df.shape[1],
    "train_default_rate": float(y_train.mean()),
    "test_default_rate": float(y_test.mean()),
    "preprocessor_saved": (MODELS_DIR / "preprocessor.pkl").exists(),
}

readiness_check


{'missing_values_X_train_processed': 0,
 'missing_values_X_test_processed': 0,
 'n_features_train': 79,
 'n_features_test': 79,
 'train_default_rate': 0.3,
 'test_default_rate': 0.3,
 'preprocessor_saved': True}


## Analytical conclusions

This notebook created the formal modeling layer of the project.

Key outputs:

- Engineered financial risk variables.
- Reproducible train-test split.
- Numerical scaling and categorical encoding pipeline.
- Processed datasets for supervised and unsupervised learning.
- Saved preprocessing object.
- Feature dictionary for documentation.

The project is now ready for the next notebook:

```text
notebooks/03_probability_of_default_model.ipynb
```

The next stage should build a baseline probability of default model, preferably starting with logistic regression before moving to more complex machine learning models.
